# 6.3 Q-learning과 SARSA — CliffWalking 경로 비교 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter06_3_q_sarsa_cliffwalk.ipynb)

책 본문: [6.3절](https://smhanlab.com/book-ml/kor/ml2/chapter06/3.html)

SARSA(6.2절)와 Q-learning(6.3절)이 **CliffWalking**에서 서로 다른
경로를 배우는 것을 실제로 확인합니다. 두 알고리즘의 갱신식은
**`max()` 한 줄**만 다르지만, 그 차이가 "절벽을 피하는 안전한 길"
(SARSA)과 "절벽 옆을 스치는 최단 길"(Q-learning)이라는 서로 다른
정책으로 이어집니다.


## 1. 환경 만들기: CliffWalking의 구조

`CliffWalking-v1`은 4×12 격자입니다. 아래 칸에 먼저 "절벽이 어디에
있는지"를 직접 출력해서 눈으로 확인합니다 — 시작(S)과 목표(G)는
**아래쪽(3행)**에 있고, 그 사이를 **절벽(.)**이 가로막습니다.


In [ ]:
import gymnasium as gym
import random

env = gym.make("CliffWalking-v1")
n_states, n_actions = env.observation_space.n, env.action_space.n
START, GOAL = env.unwrapped.start_state_index, env.unwrapped.nS - 1
rows, cols = env.unwrapped.shape
# cliff = "어떤 행동을 해도 reward -100으로 절벽 아래로"인 칸 (환경이 직접 제공하는 _cliff)
cliff = {r*cols+c for r in range(rows) for c in range(cols) if env.unwrapped._cliff[r][c]}
print(f"격자 {rows}×{cols}, 상태 {n_states}개, 행동 {n_actions}개")
print(f"시작 S = {START} (행 {START//cols}, 열 {START%cols})")
print(f"목표 G = {GOAL} (행 {GOAL//cols}, 열 {GOAL%cols})")
print(f"절벽 {len(cliff)}개 (행 {min(c//cols for c in cliff)}~{max(c//cols for c in cliff)})")
print()
for r in range(rows):
    line = ""
    for c in range(cols):
        s = r*cols+c
        line += "S" if s==START else "G" if s==GOAL else "." if s in cliff else "_"
    print(f"  {r}행: {line}")


**핵심 관찰:** 시작(S)과 목표(G)가 같은 행(3행)에 있고, 그 사이
(열 1~10)가 절벽입니다. 따라서 "짧은 길"은 **절벽 바로 위(2행)**를
스치는 길이고, "긴 길"은 절벽에서 떨어질 수 없는 **위쪽(0~1행)**을
오래도록 우회하는 길입니다.


## 2. SARSA와 Q-learning 구현 (책 6.2~6.3절 코드)

두 알고리즘은 **`max()` 한 줄**만 다릅니다. SARSA는 실제로 고른 다음
행동 `na`의 Q값을, Q-learning은 `max(Q[ns])`를 목표값에 씁니다.


In [ ]:
def epsilon_greedy(Q, s, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

def train(mode, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    random.seed(seed)
    Q = [[0.0]*n_actions for _ in range(n_states)]
    returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        a = epsilon_greedy(Q, s, epsilon, n_actions)
        tot, steps = 0.0, 0
        for _ in range(500):
            ns, r, term, trunc, _ = env.step(a)
            tot += r; steps += 1
            done = term or trunc
            na = epsilon_greedy(Q, ns, epsilon, n_actions)
            if mode == "q":
                tgt = r + (gamma*max(Q[ns]) if not done else 0.0)   # <-- max (off-policy)
            else:
                tgt = r + (gamma*Q[ns][na] if not done else 0.0)   # <-- 실제로 고른 na (on-policy)
            Q[s][a] += alpha*(tgt - Q[s][a])
            s, a = ns, na
            if done:
                break
        returns.append((tot, steps))
    return Q, returns

Q_sarsa, rets_sarsa = train("sarsa")
Q_q,     rets_q     = train("q")
print("학습 완료 (각 500 에피소드, alpha=0.5, gamma=1.0, epsilon=0.1)")


## 3. 두 숫자를 다른 방향으로 읽기

- **학습 중(탐험 포함) 마지막 100 에피소드 평균 리턴**: SARSA가 더
  좋다(절벽 추락을 피하기 때문).
- **학습 후 탐욕적(\(\varepsilon=0\)) 실행 리턴**: Q-learning이 더
  좋다(더 짧은 경로를 찾았기 때문).


In [ ]:
def mean_last100(rets):
    return sum(r for r,_ in rets[-100:])/100

print("학습 중(탐험 포함) 마지막 100 에피소드 평균 리턴:")
print(f"  SARSA:      {mean_last100(rets_sarsa):.2f}")
print(f"  Q-learning: {mean_last100(rets_q):.2f}")
print()

def greedy_rollout(Q, n_ep=100, seed=0):
    random.seed(seed)
    rets, steps_list = [], []
    for ep in range(n_ep):
        s,_ = env.reset(seed=ep); tot, steps = 0.0, 0
        for _ in range(500):
            a = max(range(n_actions), key=lambda a: Q[s][a])   # 탐욕적 (no exploration)
            ns,r,term,trunc,_ = env.step(a); tot+=r; steps+=1
            s=ns
            if term or trunc: break
        rets.append(tot); steps_list.append(steps)
    return sum(rets)/n_ep, sum(steps_list)/n_ep

g_ret_sarsa, g_steps_sarsa = greedy_rollout(Q_sarsa)
g_ret_q,     g_steps_q     = greedy_rollout(Q_q)
print("학습 후 탐욕적(epsilon=0) 실행 (100 에피소드 평균):")
print(f"  SARSA:      return {g_ret_sarsa:.1f}  (평균 {g_steps_sarsa:.0f} 스텝)")
print(f"  Q-learning: return {g_ret_q:.1f}  (평균 {g_steps_q:.0f} 스텝)")


## 4. 배운 경로를 격자에 그리기 (그림)

각 정책의 **탐욕적 경로**를 한 번 추적해서 격자에 그립니다.
절벽(검정), S·G 표시, 두 경로를 색으로 구분합니다.


In [ ]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np
# 한국어 라벨을 위한 CJK 폰트 (없으면 DejaVu Sans로 fallback)
kr=[f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"]=[kr[0]]
plt.rcParams["axes.unicode_minus"]=False

def greedy_path(Q, seed=0):
    random.seed(seed)
    s,_ = env.reset(seed=0); path=[s]
    for _ in range(500):
        a = max(range(n_actions), key=lambda a: Q[s][a])
        ns,r,term,trunc,_ = env.step(a); s=ns; path.append(s)
        if term or trunc: break
    return path

path_sarsa = greedy_path(Q_sarsa)
path_q     = greedy_path(Q_q)
print(f"SARSA 탐욕적 경로: {len(path_sarsa)-1} 스텝, 사용 행 = {sorted(set(x//cols for x in path_sarsa))}")
print(f"Q-learning 탐욕적 경로: {len(path_q)-1} 스텝, 사용 행 = {sorted(set(x//cols for x in path_q))}")


In [ ]:
def draw_cell(ax, s, color, edgecolor='none', zorder=2, alpha=1.0):
    r,c = s//cols, s%cols
    ax.add_patch(plt.Rectangle((c, rows-1-r), 1, 1, facecolor=color, edgecolor=edgecolor, zorder=zorder, alpha=alpha))

fig, ax = plt.subplots(figsize=(10,4))
# base
for s in range(n_states):
    col = "#1a1a1a" if s in cliff else "white"
    draw_cell(ax, s, col, edgecolor='gray', zorder=1)
# paths (draw Q-learning on top of SARSA where they differ)
# 절벽 칸은 경로에서 제외(절벽 자체를 검은색으로 표시). 경로 = 절벽에 닿지 않은 칸.
for s in path_sarsa[:-1]:
    if s not in cliff: draw_cell(ax, s, "#4a90d9", alpha=0.7, zorder=2)
for s in path_q[:-1]:
    if s not in cliff: draw_cell(ax, s, "#d9534f", alpha=0.9, zorder=3)
# start / goal
draw_cell(ax, START, "white", edgecolor='black', zorder=4)
draw_cell(ax, GOAL,  "white", edgecolor='black', zorder=4)
ax.text(cols/2, 0.35, "S", ha='center', va='center', fontsize=14, color='black')
ax.text(cols/2, rows-0.35, "G", ha='center', va='center', fontsize=14, color='black')
ax.set_xlim(-0.1, cols+0.1); ax.set_ylim(-0.1, rows+0.1)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("CliffWalking: SARSA(파랑, 위쪽 우회) vs Q-learning(빨강, 절벽 옆 최단)", fontsize=11)
# legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#4a90d9", alpha=0.6, label=f"SARSA ({len(path_sarsa)-1}스텝, 안전)"),
    Patch(facecolor="#d9534f", label=f"Q-learning ({len(path_q)-1}스텝, 최단)"),
    Patch(facecolor="#1a1a1a", label="절벽"),
], loc='lower right', fontsize=9)
fig.tight_layout()
fig.savefig("ch06_3_cliff_paths.svg", bbox_inches='tight')
plt.show()
print("그림 저장: ch06_3_cliff_paths.svg")


## 5. 최소 예시: "같은 경험, 다른 목표값"

CliffWalking이 크다면, **두 상태**의 장난감 환경으로 "SARSA와
Q-learning이 같은 경험에서 다른 목표를 만든다"는 점을 가장 작게
확인합니다. 상태 0에서 두 행동: `safe`(항상 0으로, 보상 -1)와
`risk`(0.9 확률로 상태 2(목표, +10)로, 0.1 확률로 0으로 보상 -1).
\(\varepsilon\)-greedy로 행동하면:

- **Q-learning**(\(\max\)을 씀)은 "다음에서 최선을 고릴 테니"
  `risk`의 가치를 낙관적으로 높게 평가해 **`risk`를 선호**합니다.
- **SARSA**(실제로 고른 것을 씀)은 무작위로 `safe`를 고를
  가능성을 반영해 `risk`의 가치를 더 보수적으로 만듭니다.


In [ ]:
gamma=0.9; eps=0.3; alpha=0.05; n_ep=200000

def demo_train(mode, seed=1):
    random.seed(seed)
    Q={0:[0.0,0.0],2:[0.0,0.0]}   # state0: [safe,risk]; state2: 목표
    for _ in range(n_ep):
        s=0
        a=max(range(2),key=lambda a:Q[s][a]) if random.random()>=eps else random.randrange(2)  # epsilon-greedy
        for _ in range(50):
            if s==2: r,ns=0.0,2
            elif a==1:   # risk
                if random.random()<0.9: r,ns=10.0,2
                else: r,ns=-1.0,0
            else:        # safe
                r,ns=-1.0,0
            done=(ns==2)
            na=max(range(2),key=lambda a:Q[ns][a]) if random.random()>=eps else random.randrange(2)
            if mode=="q": tgt=r+(gamma*max(Q[ns]) if not done else 0.0)
            else: tgt=r+(gamma*Q[ns][na] if not done else 0.0)
            Q[s][a]+=alpha*(tgt-Q[s][a])
            s,a=ns,na
            if done: break
    return Q

Qq=demo_train("q"); Qs=demo_train("sarsa")
print("최소 예시 (epsilon=0.3, 20만 에피소드):")
print(f"  Q-learning: Q(safe)={Qq[0][0]:.2f}  Q(risk)={Qq[0][1]:.2f}  -> argmax={'risk' if Qq[0][1]>Qq[0][0] else 'safe'}")
print(f"  SARSA:      Q(safe)={Qs[0][0]:.2f}  Q(risk)={Qs[0][1]:.2f}  -> argmax={'risk' if Qs[0][1]>Qs[0][0] else 'safe'}")
print()
print("Q-learning이 risk를 더 높게(낙관적으로) 평가함을 볼 수 있습니다 —")
print("'다음에서 최선을 고릴 테니'라는 가정이 risk의 가치를 끌어올리기 때문입니다.")


## 정리

- SARSA(6.2절)와 Q-learning(6.3절)의 갱신식은 **`max()` 한 줄**만
  다릅니다. 그 차이가 on-policy(행동 정책의 가치)와
  off-policy(최적 정책의 가치)를 가릅니다.
- **학습 중**에는 SARSA가(절벽을 피해) 성적이 좋고, **학습 후
  탐욕적 실행**에서는 Q-learning이(더 짧은 길) 성적이 좋습니다.
  어느 쪽이 "더 낫다"가 아니라, **어떤 질문**(행동 정책 vs 목표
  정책)을 하고 있느냐의 문제입니다.
- 실전 선택: 학습 도중 안전이 중요하다면 SARSA류, 시뮬레이터에서
  학습한 뒤 최종 성능만 중요하다면 Q-learning.
